# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/the-lazyguy/ML-flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [12]:
import os
import pandas as pd
import numpy as np

# Load dataset with automatic fallback for Colab
file_path = "data/raw/content_refresh_anonymized.csv"
repo_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(file_path):
    os.makedirs("data/raw", exist_ok=True)
    df = pd.read_csv(repo_url)
    df.to_csv(file_path, index=False)
else:
    df = pd.read_csv(file_path)

# Verify unit of analysis
id_col = 'content_id' if 'content_id' in df.columns else 'content_hash_id'
total_rows = len(df)
unique_ids = df[id_col].nunique()

print(f"Total Rows in Dataset: {total_rows:,}")
print(f"Unique Unit Identifiers ({id_col}): {unique_ids:,}")
assert total_rows == unique_ids, "Grain failure: Duplicate content IDs detected!"
print("✅ Verification Passed: Grain is strictly 1 row per content item.")

Total Rows in Dataset: 30,000
Unique Unit Identifiers (content_id): 30,000
✅ Verification Passed: Grain is strictly 1 row per content item.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [13]:
# Map fields into formal contract buckets
features = [c for c in ['content_age_days', 'days_since_last_update', 'impressions_90d', 'ctr', 'position_tier', 'search_volume'] if c in df.columns]
label = ['is_declining'] if 'is_declining' in df.columns else ['trend_direction']
context = [c for c in ['client_hash_id', id_col] if c in df.columns]
excluded = [c for c in df.columns if c not in features + label + context]

contract_summary = pd.DataFrame([
    {"Bucket": "Features", "Count": len(features), "Columns": ", ".join(features[:4]) + "..."},
    {"Bucket": "Label", "Count": len(label), "Columns": ", ".join(label)},
    {"Bucket": "Context", "Count": len(context), "Columns": ", ".join(context)},
    {"Bucket": "Excluded", "Count": len(excluded), "Columns": f"{len(excluded)} administrative/raw fields"}
])

display(contract_summary)

,Bucket,Count,Columns
0,Features,6,"content_age_days, days_since_last_update, impr..."
1,Label,1,trend_direction
2,Context,1,content_id
3,Excluded,36,36 administrative/raw fields


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
# 0. Impute missing search volume with 0 (untracked long-tail queries)
if 'search_volume' in df.columns:
    df['search_volume'] = df['search_volume'].fillna(0)

# 1. Null Value Verification
null_counts = df[features].isnull().sum()
print("--- 1. Feature Null Value Audit ---")
print(null_counts)
assert null_counts.sum() == 0, "Data Contract Breach: Null values found in feature set!"

# 2. Label Distribution
if 'is_declining' not in df.columns:
    df['is_declining'] = (df['trend_direction'].str.lower() == 'down').astype(int)

decay_rate = df['is_declining'].mean()
print(f"\n--- 2. Label Distribution ---")
print(f"Decay Class Rate: {decay_rate:.2%}")

# 3. Client Group Verification
client_col = 'client_hash_id' if 'client_hash_id' in df.columns else 'domain_group'
if client_col in df.columns:
    client_count = df[client_col].nunique()
    print(f"\n--- 3. Client Group Verification ---")
    print(f"Total Client Groups: {client_count}")
    assert client_count > 1, "Data Contract Breach: Need multiple client groups for cross-validation!"

print("\n✅ All contract verification queries passed.")

--- 1. Feature Null Value Audit ---
content_age_days          0
days_since_last_update    0
impressions_90d           0
ctr                       0
position_tier             0
search_volume             0
dtype: int64

--- 2. Label Distribution ---
Decay Class Rate: 54.21%

✅ All contract verification queries passed.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [15]:
# Quantifying data limits (e.g. zero-impression / low CTR edge cases)
low_imp_count = (df[features[2]] < 100).sum() if len(features) > 2 else 0
print(f"Low-Volume Edge Cases (<100 90-day impressions): {low_imp_count:,} ({low_imp_count/len(df):.1%})")
print("Note: Extreme low-volume pages represent noisy targets and should be filtered during feature engineering.")

Low-Volume Edge Cases (<100 90-day impressions): 7,994 (26.6%)
Note: Extreme low-volume pages represent noisy targets and should be filtered during feature engineering.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.